# Session VII: Command Line Tools and Automation

Use Python to automate tasks and interact through the command line, including parsing arguments and running scripts from the terminal.

## Exercise 1: CLI Greeter and Calculator
**Objective:** Practice basic command-line parsing using a list that represents `sys.argv`.

Instead of reading real terminal input in the notebook, you will write a function that receives an argument list like:

```python
["greet", "Alice"]
["add", "3", "4"]
```

**Tasks:**
1. `run_command(argv)` should support two commands:
   - `greet <name>` → return `"Hello, <name>!"`
   - `add <x> <y>` → return the numeric sum as a string
2. Return a helpful usage message for missing or invalid input.
3. Use `try/except` to handle non-numeric values in `add`.


In [2]:
def run_command(argv):
    """Parse a simple command list and return a response string."""
    # TODO: Handle empty input
    # TODO: Support: greet <name>
    # TODO: Support: add <x> <y>
    # TODO: Use try/except for numeric conversion
    # TODO: Return an error/usage message for bad commands
    def greet(name):
        return f"Hello, {name}!"

    def add(x, y):
        return float(x) + float(y)

    if not argv:
        return "Error: No command provided."

    command = argv[0]

    if command == "greet":
        if len(argv) != 2:
            return "Error: greet command requires exactly one argument."
        return greet(argv[1])
    elif command == "add":
        if len(argv) != 3:
            return "Error: add command requires exactly two arguments."
        try:
            return add(argv[1], argv[2])
        except ValueError:
            return "Error: Both arguments for add must be numbers."
    else:
        return "Error: Unknown command."

# --- TEST ---
print(run_command(["greet", "Alice"]))
# Expected: Hello, Alice!

print(run_command(["add", "3", "4"]))
# Expected: 7.0

print(run_command(["add", "three", "4"]))
# Expected: Error message about numeric input


Hello, Alice!
7.0
Error: Both arguments for add must be numbers.


---
## Exercise 2: Expense Lister with `argparse`
**Objective:** Build a real argument parser using optional flags.

You are given a list of expense records. Build a command-line style filter that can:
- show all expenses
- filter by category
- filter by a minimum amount

**Tasks:**
1. `build_parser()` → Create an `ArgumentParser` with:
   - `--category`
   - `--min` (float, default `0.0`)
2. `filter_expenses(expenses, category=None, min_amount=0)` → Return only matching expenses.
3. `format_expenses(expenses)` → Return a formatted multi-line string.


In [3]:
import argparse

expenses = [
    {"date": "2026-10-01", "category": "Food", "amount": 18.50, "note": "Lunch"},
    {"date": "2026-10-02", "category": "Transport", "amount": 2.80, "note": "Metro"},
    {"date": "2026-10-02", "category": "Food", "amount": 34.20, "note": "Groceries"},
    {"date": "2026-10-03", "category": "Books", "amount": 45.00, "note": "Python book"},
]

def build_parser():
    """Return an argparse parser for expense filtering."""
    # TODO: Create ArgumentParser
    # TODO: Add optional --category argument
    # TODO: Add optional --min argument (type=float, default=0.0)
    parser = argparse.ArgumentParser(description="Filter expenses by category and minimum amount.")
    parser.add_argument("--category", type=str, help="Filter by category")
    parser.add_argument("--min", type=float, default=0.0, help="Filter by minimum amount")
    return parser

def filter_expenses(expenses, category=None, min_amount=0):
    """Return filtered expenses matching the given criteria."""
    # TODO: Loop through expenses and keep only matching rows
    filtered = []
    for expense in expenses:
        if category and expense["category"] != category:
            continue
        if expense["amount"] < min_amount:
            continue
        filtered.append(expense)
    return filtered

def format_expenses(expenses):
    """Return a formatted text report for a list of expenses."""
    # TODO: Handle empty list
    # TODO: Build a list of output lines
    # TODO: Return one multi-line string
    if not expenses:
        return "No expenses found."
    lines = []
    for expense in expenses:
        line = f"{expense['date']} | {expense['category']} | ${expense['amount']:.2f} | {expense['note']}"
        lines.append(line)
    return "\n".join(lines)


# --- TEST ---
parser = build_parser()
args = parser.parse_args(["--category", "Food", "--min", "20"])
filtered = filter_expenses(expenses, args.category, args.min)
print(format_expenses(filtered))


2026-10-02 | Food | $34.20 | Groceries


---
## Exercise 3: Budget Backup Script
**Objective:** Automate data backups using `pathlib`, timestamps, and file copying.

Your script should validate that the source file exists, create a `backups` folder if needed, and make a timestamped copy.

**Tasks:**
1. `make_backup_name(filename)` → Return a new filename like `budget_data_20261005_224700.json` for `budget_data.json`
2. `backup_file(source_path, backup_dir="backups")` → Copy the source file into the backup folder.
3. Return the destination path as a string, or `None` if the source file is missing.

**First, run the setup cell below to create a sample file.**


In [4]:
from pathlib import Path
import json

sample_budget = [
    {"category": "Food", "amount": 20.5},
    {"category": "Transport", "amount": 15.0},
]

with open("budget_data.json", "w", encoding="utf-8") as file:
    json.dump(sample_budget, file, indent=2)

print("Created budget_data.json")


Created budget_data.json


In [5]:
from pathlib import Path
from datetime import datetime
import shutil

def make_backup_name(filename):
    """Return a timestamped backup filename."""
    # TODO: Use Path(filename).stem and .suffix
    # TODO: Add a timestamp with datetime.now().strftime(...)
    path = Path(filename)
    timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
    backup_name = f"{path.stem}_{timestamp}{path.suffix}"
    return backup_name

def backup_file(source_path, backup_dir="backups"):
    """Copy a file into backup_dir with a timestamped name."""
    # TODO: Convert to Path objects
    # TODO: Check whether source exists
    # TODO: Create backup folder if needed
    # TODO: Copy file with shutil.copy2
    # TODO: Return destination path as a string
    source = Path(source_path)
    if not source.exists():
        raise FileNotFoundError(f"Source file {source} does not exist.")
    backup_dir_path = Path(backup_dir)
    backup_dir_path.mkdir(parents=True, exist_ok=True)
    backup_name = make_backup_name(source.name)
    destination = backup_dir_path / backup_name
    shutil.copy2(source, destination)
    return str(destination)


# --- TEST ---
result = backup_file("budget_data.json")
print("Backup created:", result)


Backup created: backups/budget_data_20261110124913.json


---
## Exercise 4: CSV Expense Summary CLI
**Objective:** Load a CSV file, validate rows, and print category totals.

**Tasks:**
1. `load_expenses_csv(filename)` → Use `csv.DictReader` to load rows.
2. Skip malformed rows where:
   - `category` is empty
   - `amount` is missing or non-numeric
3. `summarize_by_category(expenses)` → Return a dictionary of totals.
4. `print_summary(summary)` → Print a formatted report.

**Run the setup cell below first.**


In [6]:
import csv

rows = [
    ["date", "category", "amount", "note"],
    ["2026-10-01", "Food", "18.50", "Lunch"],
    ["2026-10-02", "Transport", "2.80", "Metro"],
    ["2026-10-02", "", "12.00", "Broken row"],
    ["2026-10-03", "Books", "forty", "Invalid amount"],
    ["2026-10-04", "Food", "21.30", "Dinner"],
]

with open("expenses_cli.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerows(rows)

print("Created expenses_cli.csv")


Created expenses_cli.csv


In [7]:
import csv

def load_expenses_csv(filename):
    """Load valid expense rows from a CSV file."""
    # TODO: Use csv.DictReader
    # TODO: Validate category and amount
    # TODO: Convert amount to float
    # TODO: Skip bad rows safely
    expenses = []
    with open(filename, newline="", encoding="utf-8") as file:
        reader = csv.DictReader(file)
        for row in reader:
            category = row.get("category", "").strip()
            amount_str = row.get("amount", "").strip()
            if not category:
                continue  # Skip rows with empty category
            try:
                amount = float(amount_str)
            except ValueError:
                continue  # Skip rows with invalid amount
            expenses.append({
                "date": row.get("date", "").strip(),
                "category": category,
                "amount": amount,
                "note": row.get("note", "").strip(),
            })
    return expenses

def summarize_by_category(expenses):
    """Return {category: total_amount}."""
    # TODO: Accumulate totals in a dictionary
    summary = {}
    for expense in expenses:
        category = expense["category"]
        amount = expense["amount"]
        if category in summary:
            summary[category] += amount
        else:
            summary[category] = amount
    return summary

def print_summary(summary):
    """Print a formatted category summary."""
    # TODO: Handle empty summary
    # TODO: Print aligned output
    if not summary:
        print("No expenses to summarize.")
        return

    print("Category Summary:")
    print("-" * 40)
    for category, total in summary.items():
        print(f"{category:<20} | ${total:>10.2f}")


# --- TEST ---
expenses = load_expenses_csv("expenses_cli.csv")
summary = summarize_by_category(expenses)
print_summary(summary)


Category Summary:
----------------------------------------
Food                 | $     39.80
Transport            | $      2.80


---
## Exercise 5: Scheduling Helper
**Objective:** Generate schedule strings for cron and Windows Task Scheduler.

You are **not** actually scheduling tasks in this notebook. Instead, you will generate the correct command strings.

**Tasks:**
1. `build_cron_schedule(minute, hour, script_path)` → Return a cron line like:
   `30 18 * * * python /path/to/script.py`

   (The exercise uses plain `python` to keep the strings short; on a real machine use the full path to your project's interpreter, as shown in the slides.)
2. `build_task_scheduler_command(time_str, script_path)` → Return a Windows-style command string.
3. Validate the time values and raise or handle `ValueError` for invalid input.


In [8]:
def build_cron_schedule(minute, hour, script_path):
    """Return a cron schedule line for a daily script."""
    # TODO: Validate minute (0-59) and hour (0-23)
    # TODO: Return the cron line as a string
    if not (0 <= minute <= 59):
        raise ValueError("Minute must be between 0 and 59.")
    if not (0 <= hour <= 23):
        raise ValueError("Hour must be between 0 and 23.")
    return f"{minute} {hour} * * * python {script_path}"    

def build_task_scheduler_command(time_str, script_path):
    """Return a Windows schtasks command string."""
    # TODO: Validate HH:MM format
    # TODO: Return a command like:
    # schtasks /Create /SC DAILY /TN BudgetBackup /TR "python script.py" /ST 18:30
    import re

    if not re.match(r"^\d{2}:\d{2}$", time_str):
        raise ValueError("Time must be in HH:MM format.")

    return f"schtasks /Create /SC DAILY /TN BudgetBackup /TR \"python {script_path}\" /ST {time_str}"


# --- TEST ---
print(build_cron_schedule(30, 18, "/home/user/backup.py"))
print(build_task_scheduler_command("18:30", "C:/scripts/backup.py"))


30 18 * * * python /home/user/backup.py
schtasks /Create /SC DAILY /TN BudgetBackup /TR "python C:/scripts/backup.py" /ST 18:30


---
## Exercise 6: Automation Suite
**Objective:** Combine parsing, loading JSON, listing expenses, backups, and summaries into one mini CLI application.

Build a tool with these subcommands:
- `list` → print all expenses
- `summary` → show totals by category
- `backup` → make a timestamped backup of the JSON file

**Tasks:**
1. `load_json_expenses(filename)` → Safely load a JSON list of expense dictionaries.
2. `build_suite_parser()` → Create an `argparse` parser with subcommands:
   - `list`
   - `summary`
   - `backup`
   Also add a common `filename` argument. It comes **before** the subcommand: `["expenses_suite.json", "list"]`.
3. `run_suite(argv=None)` → Parse the arguments and perform the correct action.
4. Reuse functions from earlier exercises when possible.

**Run the setup cell below first.**


In [9]:
import json

suite_data = [
    {"date": "2026-10-01", "category": "Food", "amount": 18.50, "note": "Lunch"},
    {"date": "2026-10-02", "category": "Transport", "amount": 2.80, "note": "Metro"},
    {"date": "2026-10-03", "category": "Food", "amount": 21.30, "note": "Dinner"},
    {"date": "2026-10-04", "category": "Books", "amount": 45.00, "note": "Python book"},
]

with open("expenses_suite.json", "w", encoding="utf-8") as file:
    json.dump(suite_data, file, indent=2)

print("Created expenses_suite.json")


Created expenses_suite.json


In [14]:
import argparse
import json

def load_json_expenses(filename):
    """Safely load and validate a JSON expense list."""
    # TODO: Handle FileNotFoundError and json.JSONDecodeError
    # TODO: Ensure the top-level object is a list
    # TODO: Return [] on error
    try:
        with open(filename, "r", encoding="utf-8") as file:
            data = json.load(file)
            if not isinstance(data, list):
                print(f"Error: JSON data in {filename} is not a list.")
                return []
            return data
    except FileNotFoundError:
        print(f"Error: File {filename} not found.")
        return []
    except json.JSONDecodeError:
        print(f"Error: File {filename} contains invalid JSON.")
        return []

def build_suite_parser():
    """Return an argparse parser with subcommands."""
    # TODO: Create parser
    # TODO: Add a required filename argument
    # TODO: Add subcommands: list, summary, backup
    parser = argparse.ArgumentParser(description="Expense Automation Suite")
    parser.add_argument("filename", help="Path to the JSON expense file.")
    subparsers = parser.add_subparsers(dest="command", help="Available commands.")

    list_parser = subparsers.add_parser("list", help="List all expenses.")
    summary_parser = subparsers.add_parser("summary", help="Show a summary of expenses by category.")
    backup_parser = subparsers.add_parser("backup", help="Create a backup of the expense file.")

    return parser

def run_suite(argv):
    """Run the automation suite and return a message or report."""
    # TODO: Parse args
    # TODO: Load data when needed
    # TODO: Route to list / summary / backup
    parser = build_suite_parser()
    args = parser.parse_args(argv)

# --- TEST ---
print(run_suite(["expenses_suite.json", "list"]))
print()
print(run_suite(["expenses_suite.json", "summary"]))
print()
print(run_suite(["expenses_suite.json", "backup"]))


None

None

None
